# 예제 03. MLP 모델 학습
빅데이터프로그래밍 · 7주차

## 목표
- MLP 모델을 작성한다
- MNIST로 학습시킨다
- 학습 손실과 정확도를 epoch마다 기록한다

5주차 학습 루프와 6주차 모델 클래스를 그대로 씁니다. 데이터만 실제 이미지로 바뀌었습니다.

**런타임 > 런타임 유형 변경 > T4 GPU** 를 먼저 선택하세요.


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)


## 1. 데이터 준비


In [ ]:
transform = transforms.ToTensor()
train_set = datasets.MNIST("./data", train=True,  download=True, transform=transform)
test_set  = datasets.MNIST("./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_set,  batch_size=256, shuffle=False)

print("학습 batch 수:", len(train_loader), "/ 시험 batch 수:", len(test_loader))


## 2. 모델 작성
784 → 128 → 64 → 10


In [ ]:
class MLP(nn.Module):
    def __init__(self, hidden1=128, hidden2=64, n_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(28 * 28, hidden1)
        self.fc2 = nn.Linear(hidden1, hidden2)
        self.fc3 = nn.Linear(hidden2, n_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        return self.fc3(x)


model = MLP().to(device)
print(model)
print("파라미터:", sum(p.numel() for p in model.parameters()))


## 3. 학습 전 확인 — 임의 입력으로 shape 점검


In [ ]:
dummy = torch.randn(4, 1, 28, 28).to(device)
print("입력:", tuple(dummy.shape), "→ 출력:", tuple(model(dummy).shape))


## 4. 학습


In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

def evaluate(loader):
    model.eval()
    loss_sum = correct = total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss_sum += loss_fn(out, y).item() * y.numel()
            correct += (out.argmax(dim=1) == y).sum().item()
            total += y.numel()
    return loss_sum / total, correct / total


history = []
epochs = 10

for epoch in range(1, epochs + 1):
    model.train()
    loss_sum = correct = total = 0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)

        out = model(x)
        loss = loss_fn(out, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        loss_sum += loss.item() * y.numel()
        correct += (out.argmax(dim=1) == y).sum().item()
        total += y.numel()

    train_loss, train_acc = loss_sum / total, correct / total
    test_loss, test_acc = evaluate(test_loader)
    history.append((train_loss, train_acc, test_loss, test_acc))

    print(f"epoch {epoch:2d}  train loss {train_loss:.4f} acc {train_acc:.4f}  "
          f"|  test loss {test_loss:.4f} acc {test_acc:.4f}")


## 5. 학습 곡선


In [ ]:
import matplotlib.pyplot as plt

tr_l = [h[0] for h in history]; tr_a = [h[1] for h in history]
te_l = [h[2] for h in history]; te_a = [h[3] for h in history]
xs = range(1, len(history) + 1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(xs, tr_l, label="train"); ax[0].plot(xs, te_l, label="test")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(xs, tr_a, label="train"); ax[1].plot(xs, te_a, label="test")
ax[1].set_title("accuracy"); ax[1].set_xlabel("epoch"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 6. 곡선 읽는 법

| 모양 | 뜻 |
| --- | --- |
| 둘 다 내려간다 | 정상적으로 학습 중 |
| 학습만 내려가고 시험은 올라간다 | 과적합 — 8주차 주제 |
| 둘 다 안 내려간다 | 학습률이나 모델 구조 문제 |


In [ ]:
gap = tr_a[-1] - te_a[-1]
print(f"학습 정확도 {tr_a[-1]:.4f} / 시험 정확도 {te_a[-1]:.4f}")
print(f"차이 {gap:.4f}", "→ 과적합 경향" if gap > 0.03 else "→ 아직 괜찮습니다")


## 7. 모델 저장
다음 노트북에서 다시 쓰려면 저장해 둡니다.


In [ ]:
torch.save(model.state_dict(), "mnist_mlp.pt")
print("저장 완료")


## 직접 해보기
1. epoch을 20으로 늘리면 시험 정확도는 어떻게 되나요?
2. `Adam` 을 `SGD(lr=0.1)` 로 바꿔 학습 곡선을 비교하세요.


In [ ]:
# 여기에 작성하세요
